In [19]:
import random
from torchvision import datasets, transforms
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F
import torch.utils.data as Data
import matplotlib.pyplot as plt
import math
import scienceplots
import matplotlib as mpl
from tqdm import tqdm
import optuna
SEED = 1234
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# 定义数据
x = torch.unsqueeze(torch.linspace(-2, 10, 19), dim=1)
ytrain = np.exp(-(x - 2) ** 2) + np.exp(-(x - 6) ** 2 / 10) + 1 / (x ** 2 + 1) + 0.12 * torch.randn(x.size())

class CustomActivation(nn.Module):  # 激活函数含噪声参数
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, input):
        temp = 1/2 + torch.erf(input / (math.sqrt(2) * self.sigma))/2   # 前向传播
        return temp


class Net(nn.Module):  # 定义网络
    def __init__(self, n_feature, n_hidden1, n_output, sigma):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(n_feature, n_hidden1)  # 全连接层
        self.custom1 = CustomActivation(sigma)  # 自定义激活函数层
        self.predict = nn.Linear(n_hidden1, n_output)  # 输出层

    def forward(self, x):
        x = self.hidden1(x)
        x = self.custom1(x)
        x = self.predict(x)  # 前向传播过程
        return x

    def reset_parameters(self):
        self.hidden1.reset_parameters()
        self.predict.reset_parameters()


device = torch.device('cpu')
net = Net(n_feature=1, n_hidden1=14, n_output=1, sigma=5.65)
optimizer = optim.Adam(net.parameters(), lr=0.01, betas=(0.99, 0.99))
loss_func = torch.nn.MSELoss().to(device)

NJnum = 1
epoch = 15000
x_respond = torch.unsqueeze(torch.linspace(-2, 10, 300), dim=1)

train_losses = []
responses = []

# 训练循环
for num in range(NJnum):
    seed = random.sample(range(1, 10000), 1)[0]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    net.reset_parameters()

    for _ in range(epoch):
        net.train()
        prediction = net(x)
        loss = loss_func(prediction, ytrain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        response = net(x_respond)
        responses.append(response.numpy())

mean_train_loss = np.mean(train_losses)
mean_response = np.mean(responses, axis=0)

# 测试阶段
seeds = random.sample(range(1, 1000), 10)
test_losses = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    x_test = torch.unsqueeze(torch.linspace(-2, 10, 30), dim=1).to(device)
    y_test = np.exp(-(x_test - 2) ** 2) + np.exp(-(x_test - 6) ** 2 / 10) + 1 / (x_test ** 2 + 1) + 0.12 * torch.randn(
        x_test.size()).to(device)

    net.eval()
    with torch.no_grad():
        prediction_test = net(x_test)
        test_loss = loss_func(prediction_test, y_test)
        test_losses.append(test_loss.item())

mean_test_loss = np.mean(test_losses)
std_test_loss = np.std(test_losses)
print("Mean Train Loss:", mean_train_loss)
print("Mean Test Loss:", mean_test_loss)
print(std_test_loss)

Mean Train Loss: 0.010163052938878536
Mean Test Loss: 0.019491089042276144
0.00654302073349335


In [23]:
import random
from torchvision import datasets, transforms
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F
import torch.utils.data as Data
import matplotlib.pyplot as plt
import math
import scienceplots
import matplotlib as mpl
from tqdm import tqdm
import optuna
SEED = 1234
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# 定义数据
x = torch.unsqueeze(torch.linspace(-2, 10, 19), dim=1)
ytrain = np.exp(-(x - 2) ** 2) + np.exp(-(x - 6) ** 2 / 10) + 1 / (x ** 2 + 1) + 0.12 * torch.randn(x.size())

class CustomActivation(nn.Module):  # 激活函数含噪声参数
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, input):
        temp = 1/2 + torch.erf(input / (math.sqrt(2) * self.sigma))/2   # 前向传播
        return temp


class Net(nn.Module):  # 定义网络
    def __init__(self, n_feature, n_hidden1, n_output, sigma):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(n_feature, n_hidden1)  # 全连接层
        self.custom1 = CustomActivation(sigma)  # 自定义激活函数层
        self.predict = nn.Linear(n_hidden1, n_output)  # 输出层

    def forward(self, x):
        x = self.hidden1(x)
        x = self.custom1(x)
        x = self.predict(x)  # 前向传播过程
        return x

    def reset_parameters(self):
        self.hidden1.reset_parameters()
        self.predict.reset_parameters()


device = torch.device('cpu')
net = Net(n_feature=1, n_hidden1=14, n_output=1, sigma=5.65)
optimizer = optim.Adam(net.parameters(), lr=0.01, betas=(0.99, 0.99))
loss_func = torch.nn.MSELoss().to(device)

NJnum = 1
epoch = 15000
x_respond = torch.unsqueeze(torch.linspace(-2, 10, 300), dim=1)

train_losses = []
responses = []

# 训练循环
for num in range(NJnum):
    seed = random.sample(range(1, 1000), 1)[0]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    net.reset_parameters()

    for _ in range(epoch):
        net.train()
        prediction = net(x)
        loss = loss_func(prediction, ytrain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        response = net(x_respond)
        responses.append(response.numpy())

mean_train_loss = np.mean(train_losses)
mean_response = np.mean(responses, axis=0)

# 测试阶段
seeds = random.sample(range(1, 10000), 10)
test_losses = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    x_test = torch.unsqueeze(torch.linspace(-2, 10, 30), dim=1).to(device)
    y_test = np.exp(-(x_test - 2) ** 2) + np.exp(-(x_test - 6) ** 2 / 10) + 1 / (x_test ** 2 + 1) + 0.12 * torch.randn(
        x_test.size()).to(device)

    net.eval()
    with torch.no_grad():
        prediction_test = net(x_test)
        test_loss = loss_func(prediction_test, y_test)
        test_losses.append(test_loss.item())

mean_test_loss = np.mean(test_losses)
std_test_loss = np.std(test_losses)
print("Mean Train Loss:", mean_train_loss)
print("Mean Test Loss:", mean_test_loss)
print(std_test_loss)

Mean Train Loss: 0.00989173911511898
Mean Test Loss: 0.02337554795667529
0.006146929554897787


In [24]:
import random
from torchvision import datasets, transforms
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F
import torch.utils.data as Data
import matplotlib.pyplot as plt
import math
import scienceplots
import matplotlib as mpl
from tqdm import tqdm
import optuna
SEED = 1234
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# 定义数据
x = torch.unsqueeze(torch.linspace(-2, 10, 19), dim=1)
ytrain = np.exp(-(x - 2) ** 2) + np.exp(-(x - 6) ** 2 / 10) + 1 / (x ** 2 + 1) + 0.12 * torch.randn(x.size())

class CustomActivation(nn.Module):  # 激活函数含噪声参数
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, input):
        temp = 1/2 + torch.erf(input / (math.sqrt(2) * self.sigma))/2   # 前向传播
        return temp


class Net(nn.Module):  # 定义网络
    def __init__(self, n_feature, n_hidden1, n_output, sigma):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(n_feature, n_hidden1)  # 全连接层
        self.custom1 = CustomActivation(sigma)  # 自定义激活函数层
        self.predict = nn.Linear(n_hidden1, n_output)  # 输出层

    def forward(self, x):
        x = self.hidden1(x)
        x = self.custom1(x)
        x = self.predict(x)  # 前向传播过程
        return x

    def reset_parameters(self):
        self.hidden1.reset_parameters()
        self.predict.reset_parameters()


device = torch.device('cpu')
net = Net(n_feature=1, n_hidden1=14, n_output=1, sigma=5.65)
optimizer = optim.Adam(net.parameters(), lr=0.01, betas=(0.99, 0.99))
loss_func = torch.nn.MSELoss().to(device)

NJnum = 1
epoch = 15000
x_respond = torch.unsqueeze(torch.linspace(-2, 10, 300), dim=1)

train_losses = []
responses = []

# 训练循环
for num in range(NJnum):
    seed = random.sample(range(1, 20000), 1)[0]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    net.reset_parameters()

    for _ in range(epoch):
        net.train()
        prediction = net(x)
        loss = loss_func(prediction, ytrain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        response = net(x_respond)
        responses.append(response.numpy())

mean_train_loss = np.mean(train_losses)
mean_response = np.mean(responses, axis=0)

# 测试阶段
seeds = random.sample(range(1, 2000), 10)
test_losses = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    x_test = torch.unsqueeze(torch.linspace(-2, 10, 30), dim=1).to(device)
    y_test = np.exp(-(x_test - 2) ** 2) + np.exp(-(x_test - 6) ** 2 / 10) + 1 / (x_test ** 2 + 1) + 0.12 * torch.randn(
        x_test.size()).to(device)

    net.eval()
    with torch.no_grad():
        prediction_test = net(x_test)
        test_loss = loss_func(prediction_test, y_test)
        test_losses.append(test_loss.item())

mean_test_loss = np.mean(test_losses)
std_test_loss = np.std(test_losses)
print("Mean Train Loss:", mean_train_loss)
print("Mean Test Loss:", mean_test_loss)
print(std_test_loss)

Mean Train Loss: 0.010232456028461456
Mean Test Loss: 0.017969861906021834
0.003914912981277334


In [25]:
import random
from torchvision import datasets, transforms
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F
import torch.utils.data as Data
import matplotlib.pyplot as plt
import math
import scienceplots
import matplotlib as mpl
from tqdm import tqdm
import optuna
SEED = 1234
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# 定义数据
x = torch.unsqueeze(torch.linspace(-2, 10, 19), dim=1)
ytrain = np.exp(-(x - 2) ** 2) + np.exp(-(x - 6) ** 2 / 10) + 1 / (x ** 2 + 1) + 0.12 * torch.randn(x.size())

class CustomActivation(nn.Module):  # 激活函数含噪声参数
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, input):
        temp = 1/2 + torch.erf(input / (math.sqrt(2) * self.sigma))/2   # 前向传播
        return temp


class Net(nn.Module):  # 定义网络
    def __init__(self, n_feature, n_hidden1, n_output, sigma):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(n_feature, n_hidden1)  # 全连接层
        self.custom1 = CustomActivation(sigma)  # 自定义激活函数层
        self.predict = nn.Linear(n_hidden1, n_output)  # 输出层

    def forward(self, x):
        x = self.hidden1(x)
        x = self.custom1(x)
        x = self.predict(x)  # 前向传播过程
        return x

    def reset_parameters(self):
        self.hidden1.reset_parameters()
        self.predict.reset_parameters()


device = torch.device('cpu')
net = Net(n_feature=1, n_hidden1=14, n_output=1, sigma=5.65)
optimizer = optim.Adam(net.parameters(), lr=0.01, betas=(0.99, 0.99))
loss_func = torch.nn.MSELoss().to(device)

NJnum = 1
epoch = 15000
x_respond = torch.unsqueeze(torch.linspace(-2, 10, 300), dim=1)

train_losses = []
responses = []

# 训练循环
for num in range(NJnum):
    seed = random.sample(range(1, 30000), 1)[0]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    net.reset_parameters()

    for _ in range(epoch):
        net.train()
        prediction = net(x)
        loss = loss_func(prediction, ytrain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        response = net(x_respond)
        responses.append(response.numpy())

mean_train_loss = np.mean(train_losses)
mean_response = np.mean(responses, axis=0)

# 测试阶段
seeds = random.sample(range(1, 3000), 10)
test_losses = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    x_test = torch.unsqueeze(torch.linspace(-2, 10, 30), dim=1).to(device)
    y_test = np.exp(-(x_test - 2) ** 2) + np.exp(-(x_test - 6) ** 2 / 10) + 1 / (x_test ** 2 + 1) + 0.12 * torch.randn(
        x_test.size()).to(device)

    net.eval()
    with torch.no_grad():
        prediction_test = net(x_test)
        test_loss = loss_func(prediction_test, y_test)
        test_losses.append(test_loss.item())

mean_test_loss = np.mean(test_losses)
std_test_loss = np.std(test_losses)
print("Mean Train Loss:", mean_train_loss)
print("Mean Test Loss:", mean_test_loss)
print(std_test_loss)

Mean Train Loss: 0.01016333419829607
Mean Test Loss: 0.018650484550744296
0.0037945319153967066


In [ ]:
import random
from torchvision import datasets, transforms
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import numpy as np
import torch
from torch.autograd import Variable
import torch.nn.functional as F
import torch.utils.data as Data
import matplotlib.pyplot as plt
import math
import scienceplots
import matplotlib as mpl
from tqdm import tqdm
import optuna
SEED = 1234
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

# 定义数据
x = torch.unsqueeze(torch.linspace(-2, 10, 19), dim=1)
ytrain = np.exp(-(x - 2) ** 2) + np.exp(-(x - 6) ** 2 / 10) + 1 / (x ** 2 + 1) + 0.12 * torch.randn(x.size())

class CustomActivation(nn.Module):  # 激活函数含噪声参数
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, input):
        temp = 1/2 + torch.erf(input / (math.sqrt(2) * self.sigma))/2   # 前向传播
        return temp


class Net(nn.Module):  # 定义网络
    def __init__(self, n_feature, n_hidden1, n_output, sigma):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(n_feature, n_hidden1)  # 全连接层
        self.custom1 = CustomActivation(sigma)  # 自定义激活函数层
        self.predict = nn.Linear(n_hidden1, n_output)  # 输出层

    def forward(self, x):
        x = self.hidden1(x)
        x = self.custom1(x)
        x = self.predict(x)  # 前向传播过程
        return x

    def reset_parameters(self):
        self.hidden1.reset_parameters()
        self.predict.reset_parameters()


device = torch.device('cpu')
net = Net(n_feature=1, n_hidden1=14, n_output=1, sigma=5.65)
optimizer = optim.Adam(net.parameters(), lr=0.01, betas=(0.99, 0.99))
loss_func = torch.nn.MSELoss().to(device)

NJnum = 1
epoch = 15000
x_respond = torch.unsqueeze(torch.linspace(-2, 10, 300), dim=1)

train_losses = []
responses = []

# 训练循环
for num in range(NJnum):
    seed = random.sample(range(1, 50000), 1)[0]
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    net.reset_parameters()

    for _ in range(epoch):
        net.train()
        prediction = net(x)
        loss = loss_func(prediction, ytrain)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_losses.append(loss.item())

    net.eval()
    with torch.no_grad():
        response = net(x_respond)
        responses.append(response.numpy())

mean_train_loss = np.mean(train_losses)
mean_response = np.mean(responses, axis=0)

# 测试阶段
seeds = random.sample(range(1, 5000), 10)
test_losses = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    x_test = torch.unsqueeze(torch.linspace(-2, 10, 30), dim=1).to(device)
    y_test = np.exp(-(x_test - 2) ** 2) + np.exp(-(x_test - 6) ** 2 / 10) + 1 / (x_test ** 2 + 1) + 0.12 * torch.randn(
        x_test.size()).to(device)

    net.eval()
    with torch.no_grad():
        prediction_test = net(x_test)
        test_loss = loss_func(prediction_test, y_test)
        test_losses.append(test_loss.item())

mean_test_loss = np.mean(test_losses)
std_test_loss = np.std(test_losses)
print("Mean Train Loss:", mean_train_loss)
print("Mean Test Loss:", mean_test_loss)
print(std_test_loss)